# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import random

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: <StringArray>
[ 'United States',          'China',          'India',        'Germany',
 'United Kingdom',         'France',         'Brazil',          'Japan',
         'Canada',      'Australia',    'South Korea',         'Russia',
   'South Africa',         'Mexico',      'Indonesia']
Length: 15, dtype: str

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [24]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE

#df[df['Region'] == 'Asia']` to filter; use `go.Figure()

""" fig = px.bar(
    df[df['Region'] == 'Asia'].sort_values('CO2_Mt', ascending=True),
    x='CO2_Mt', y='Country',
    orientation='h',
    title='Co2 per Country in Asia',
    height=500
) """


highlight = random.choice(list(df[df['Region'] == 'Asia']["Country"].drop_duplicates())) #'China'

# Build colour map: highlight in blue, everything else grey
color_map = {c: 'DarkBlue' if c == highlight else 'LightGrey' for c in df['Country'].unique()}

fig = px.line(df[df['Region'] == 'Asia'], x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''})

fig.update_traces(
    line=dict(width=1.5),   # default: thin for context countries
    showlegend=False      # direct label replaces legend (Gestalt proximity)
)

# Override highlighted country: thicker line
fig.update_traces(
    line=dict(width=3),
    selector=dict(name=highlight)
)

# Direct label at end of highlighted line
last = df.loc[(df['Country'] == highlight) & (df['Year'] == df['Year'].max())]

fig.add_annotation(
    x=last['Year'].values[0], y=last['CO2_Mt'].values[0],
    text=f'<b>{highlight}</b>', showarrow=False,
    xanchor='left', xshift=6,
    font=dict(color='DarkBlue', size=12, family='Arial')
)

fig.update_layout(
    title="China's CO2 emissions tripled 2000–2022 — no other country comes close",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE', title='CO2 Emissions (Mt)'),
    xaxis=dict(showgrid=False, title=''),
    margin=dict(l=60, r=80, t=55, b=40)
)

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [39]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE
selected = list(df[df['Region'] == 'Asia']["Country"].drop_duplicates())
tg = df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()
sg = tg.loc[tg['Year'].isin([2000, 2022])].copy()

changes = sg.groupby('Region')['CO2_Mt'].agg(['first', 'last'])

color_map = {c: 'red' if row['last'] > row['first'] else 'green' for c, row in changes.iterrows()}

fig = px.line(sg.sort_values('Year'), x='Year', y='CO2_Mt', color='Region',
              color_discrete_map=color_map, markers=True,
              labels={'CO2_Mt': '', 'Year': ''})

for country in selected:
    d = sg.loc[sg['Region'] == country].sort_values('Year')
    for region in sg['Region'].unique():
        d = sg.loc[sg['Region'] == region].sort_values('Year')
        fig.update_traces(
            selector=dict(name=region),
            mode='lines+markers+text',
            text=[f'{d["CO2_Mt"].iloc[0]:.0f}', f'{region}<br>{d["CO2_Mt"].iloc[1]:.0f}'],
            textposition=['middle left', 'middle right'],
            showlegend=True
        )

fig.update_layout(
    title='Blue = increased since 2000 | Orange = decreased | China and India dwarf all others',
    xaxis=dict(tickvals=[2000, 2022], ticktext=['2000', '2022'],
               showgrid=False, range=[1995, 2028]),
    yaxis=dict(showgrid=False, showticklabels=False, title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=80, r=120, t=55, b=40),
    height=800, width=1000
)

fig.show()